# Modern Tokenizers: Special Tokens and Chat Templates

This notebook explores how modern language models use special tokens and format conversations.

We'll explore:
- GPT-4 (OpenAI) - using tiktoken
- Qwen (Alibaba) - using transformers

In [1]:
import tiktoken
from transformers import AutoTokenizer

## Part 1: Special Tokens in Modern Models

Different models use different special tokens to structure conversations.

### GPT-4 Special Tokens

In [2]:
# Load different tiktoken encodings
encodings_to_test = [
    ("cl100k_base", "GPT-4, GPT-3.5-turbo"),
    ("o200k_base", "GPT-4o, o1"),
]

for enc_name, models in encodings_to_test:
    enc = tiktoken.get_encoding(enc_name)
    print(f"\n{'='*60}")
    print(f"{enc_name} (used by: {models})")
    print('='*60)
    
    # Check if special tokens exist
    if hasattr(enc, '_special_tokens'):
        special_tokens = enc._special_tokens
        print(f"\nSpecial tokens ({len(special_tokens)} total):\n")
        
        for token, token_id in sorted(special_tokens.items(), key=lambda x: x[1]):
            print(f"  {token:<30} → ID {token_id}")
    else:
        print("\nNo _special_tokens attribute found")
    
    print(f"\nVocabulary size: {enc.n_vocab:,}")

# Keep a reference for later use
gpt4_enc = tiktoken.get_encoding("cl100k_base")


cl100k_base (used by: GPT-4, GPT-3.5-turbo)

Special tokens (5 total):

  <|endoftext|>                  → ID 100257
  <|fim_prefix|>                 → ID 100258
  <|fim_middle|>                 → ID 100259
  <|fim_suffix|>                 → ID 100260
  <|endofprompt|>                → ID 100276

Vocabulary size: 100,277

o200k_base (used by: GPT-4o, o1)

Special tokens (2 total):

  <|endoftext|>                  → ID 199999
  <|endofprompt|>                → ID 200018

Vocabulary size: 200,019


### Why Different Libraries?

**tiktoken** (OpenAI):
- OpenAI's library for their tokenizers
- Supports: GPT-3.5, GPT-4, GPT-4o, Codex, etc.
- Fast, focused library
- Does NOT work with transformers library

**transformers** (HuggingFace):
- Works with many open-source models
- Unified interface across models
- Includes: Qwen, GLM, Kimi, LLaMA, etc.
- GPT-2 tokenizer available, but NOT GPT-4 (not open-source)

**Key difference**: GPT-4 is closed-source (API only), while Qwen/GLM/Kimi have open-source tokenizers on HuggingFace.

### Qwen Special Tokens

In [3]:
# Load multiple tokenizers to compare
tokenizers_to_test = [
    ("Qwen/Qwen3-0.6B", "Qwen3 (Alibaba)"),
    ("zai-org/GLM-4.7", "GLM-4 (Zhipu AI)"),
    ("moonshotai/Kimi-K2-Thinking", "Kimi (Moonshot AI)"),
]

for model_name, display_name in tokenizers_to_test:
    print(f"\n{'='*70}")
    print(f"{display_name}")
    print(f"Model: {model_name}")
    print('='*70)
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)  
        # Show all special tokens
        if hasattr(tokenizer, 'all_special_tokens'):
            print(f"\nAll special tokens ({len(tokenizer.all_special_tokens)} total):")
            for token in tokenizer.all_special_tokens[:15]:  # Show first 15
                token_id = tokenizer.convert_tokens_to_ids(token)
                print(f"  {token:<30} → ID {token_id}")
            if len(tokenizer.all_special_tokens) > 15:
                print(f"  ... and {len(tokenizer.all_special_tokens) - 15} more")
        
        print(f"\nVocabulary size: {len(tokenizer):,}")
        
    except Exception as e:
        print(f"\nError loading tokenizer: {e}")
        print("Note: Some models may require specific access or different model paths")

# Keep a reference to Qwen for later use
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")


Qwen3 (Alibaba)
Model: Qwen/Qwen3-0.6B

All special tokens (14 total):
  <|im_end|>                     → ID 151645
  <|endoftext|>                  → ID 151643
  <|im_start|>                   → ID 151644
  <|object_ref_start|>           → ID 151646
  <|object_ref_end|>             → ID 151647
  <|box_start|>                  → ID 151648
  <|box_end|>                    → ID 151649
  <|quad_start|>                 → ID 151650
  <|quad_end|>                   → ID 151651
  <|vision_start|>               → ID 151652
  <|vision_end|>                 → ID 151653
  <|vision_pad|>                 → ID 151654
  <|image_pad|>                  → ID 151655
  <|video_pad|>                  → ID 151656

Vocabulary size: 151,669

GLM-4 (Zhipu AI)
Model: zai-org/GLM-4.7

All special tokens (22 total):
  <|endoftext|>                  → ID 151329
  [MASK]                         → ID 151330
  [gMASK]                        → ID 151331
  [sMASK]                        → ID 151332
  <sop>            

---

## Part 2: How Special Tokens Behave

### Exercise 1: Encoding vs Decoding

**Task**: What happens when you try to encode special tokens as regular text?

In [4]:
# Try encoding special token as regular text
text_with_special = "Hello <|endoftext|> world"

print("Text:", text_with_special)
print("\nGPT-4 tokenization:")
# Need to set disallowed_special=() to encode special token strings as regular text
tokens = gpt4_enc.encode(text_with_special, disallowed_special=())
print(tokens)
print("Breaking down the tokenization:")
for token_id in tokens:
    token_bytes = gpt4_enc.decode_single_token_bytes(token_id)
    token_str = token_bytes.decode('utf-8', errors='replace')
    print(f"  ID {token_id:5d} → '{token_str}'")

Text: Hello <|endoftext|> world

GPT-4 tokenization:
[9906, 83739, 8862, 728, 428, 91, 29, 1917]
Breaking down the tokenization:
  ID  9906 → 'Hello'
  ID 83739 → ' <|'
  ID  8862 → 'endo'
  ID   728 → 'ft'
  ID   428 → 'ext'
  ID    91 → '|'
  ID    29 → '>'
  ID  1917 → ' world'


In [5]:
# Special tokens are NOT treated specially by default
# They're tokenized like regular text: "<", "|", "end", "of", "text", "|", ">"

tokens_special = gpt4_enc.encode(text_with_special, allowed_special={'<|endoftext|>'})
print(tokens_special)
print("Breaking down the tokenization:")
for token_id in tokens_special:
    token_bytes = gpt4_enc.decode_single_token_bytes(token_id)
    token_str = token_bytes.decode('utf-8', errors='replace')
    print(f"  ID {token_id:5d} → '{token_str}'")
print("\nTo use special tokens, you need to insert their IDs directly!")

[9906, 220, 100257, 1917]
Breaking down the tokenization:
  ID  9906 → 'Hello'
  ID   220 → ' '
  ID 100257 → '<|endoftext|>'
  ID  1917 → ' world'

To use special tokens, you need to insert their IDs directly!


### Qwen Chat Format

In [10]:
# Example conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What is its population?"}
]

### Exercise 2: Applying Chat Template

**Task**: How do you think the model formats this conversation with special tokens?

*(Think for 30 seconds)*

In [9]:
# Apply chat template
formatted = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("Formatted conversation:")
print(formatted)

Formatted conversation:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>user
What is its population?<|im_end|>



---

### Solution 2: Understanding the Format

In [11]:
# Let's tokenize and see the token IDs
token_ids = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

print(f"Total tokens: {len(token_ids)}\n")
print("First 30 tokens:")
for i, token_id in enumerate(token_ids[:30]):
    token = qwen_tokenizer.decode([token_id])
    print(f"  {i:3d}: ID {token_id:6d} → '{token}'")

Total tokens: 45

First 30 tokens:
    0: ID 151644 → '<|im_start|>'
    1: ID   8948 → 'system'
    2: ID    198 → '
'
    3: ID   2610 → 'You'
    4: ID    525 → ' are'
    5: ID    264 → ' a'
    6: ID  10950 → ' helpful'
    7: ID  17847 → ' assistant'
    8: ID     13 → '.'
    9: ID 151645 → '<|im_end|>'
   10: ID    198 → '
'
   11: ID 151644 → '<|im_start|>'
   12: ID    872 → 'user'
   13: ID    198 → '
'
   14: ID   3838 → 'What'
   15: ID    374 → ' is'
   16: ID    279 → ' the'
   17: ID   6722 → ' capital'
   18: ID    315 → ' of'
   19: ID   9625 → ' France'
   20: ID     30 → '?'
   21: ID 151645 → '<|im_end|>'
   22: ID    198 → '
'
   23: ID 151644 → '<|im_start|>'
   24: ID  77091 → 'assistant'
   25: ID    198 → '
'
   26: ID    785 → 'The'
   27: ID   6722 → ' capital'
   28: ID    315 → ' of'
   29: ID   9625 → ' France'


### With Generation Prompt

In [12]:
# Add generation prompt (prepares for model to respond)
formatted_with_prompt = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("With generation prompt:")
print(formatted_with_prompt)
print("\nNotice the format is ready for the assistant to continue!")

With generation prompt:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
<|im_start|>user
What is its population?<|im_end|>
<|im_start|>assistant


Notice the format is ready for the assistant to continue!


---

## Part 4: GPT-4 Chat Format (Conceptual)

GPT-4 uses a similar approach through the OpenAI API, but `tiktoken` doesn't have `apply_chat_template`.

The API handles this internally:
```python
# OpenAI API format
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"}
    ]
)
```

Internally, it formats with special tokens similar to what we saw with Qwen.

---

## Part 6: Decoding Behavior with Special Tokens

In [13]:
# Decode with and without special tokens
token_ids = qwen_tokenizer.apply_chat_template(messages, tokenize=True)

print("Decoded WITH special tokens:")
decoded_with = qwen_tokenizer.decode(token_ids, skip_special_tokens=False)
print(decoded_with)
print("\n" + "="*60 + "\n")

print("Decoded WITHOUT special tokens:")
decoded_without = qwen_tokenizer.decode(token_ids, skip_special_tokens=True)
print(decoded_without)

Decoded WITH special tokens:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
<|im_start|>user
What is its population?<|im_end|>



Decoded WITHOUT special tokens:
system
You are a helpful assistant.
user
What is the capital of France?
assistant
The capital of France is Paris.
user
What is its population?



---

## Summary: Key Takeaways

1. **Special tokens structure conversations**
   - Different models use different special tokens
   - GPT-4: `<|endoftext|>`, `<|fim_*|>`, etc.
   - Qwen: `<|im_start|>`, `<|im_end|>`, etc.

2. **Chat templates format multi-turn conversations**
   - `apply_chat_template()` handles this automatically
   - Distinguishes system/user/assistant roles
   - Adds special tokens in the right places

3. **Decoding behavior**
   - `skip_special_tokens=False`: See internal structure
   - `skip_special_tokens=True`: Clean text for users

4. **Practical implications**:
   - Always check token counts for cost estimation
   - Use appropriate decoding for your use case
   - Understand your model's chat format